In [1]:
# Install required packages
!pip install -q anthropic google-generativeai requests

In [12]:
import os
import re
from datetime import datetime

# ========================================
# CHOOSE YOUR PROVIDER AND PASTE API KEY
# ========================================
PROVIDER = "gemini"  # Options: "claude" or "gemini"

CLAUDE_API_KEY = "YOUR_CLAUDE_KEY_HERE"  # Get from: https://console.anthropic.com/
GEMINI_API_KEY = os.environ["GEMINI_API_KEY"]  # Get from: https://aistudio.google.com/app/apikey

# ========================================

# Setup based on provider
if PROVIDER == "claude":
    import anthropic
    client = anthropic.Anthropic(api_key=CLAUDE_API_KEY)
    print("✅ Using Claude (Anthropic)")
    print("   Model: claude-3-haiku-20240307")
    
elif PROVIDER == "gemini":
    import google.generativeai as genai
    genai.configure(api_key=GEMINI_API_KEY)
    client = genai.GenerativeModel('models/gemini-2.5-flash')
    print("✅ Using Gemini")
    print("   ⚠️  Note: Only 20 requests/day on free tier")


✅ Using Gemini
   ⚠️  Note: Only 20 requests/day on free tier


# Baseline Prompt

In [ ]:
import pandas as pd

def call_llm(df, Turb_ID, base_day):
    turbine_data = df[df['TurbID'] == Turb_ID]
    # Drop rows where 'Wspd' (Wind Speed) is missing
    turbine_data = turbine_data.dropna(subset=['Wspd'])

    day_data = turbine_data[turbine_data['Day'] == base_day].drop(columns=['Day', 'TurbID'])

    data_str = day_data.to_csv(index=False, sep=',')

    prompt = f"""
Context: You are a energy forecaster for wind turbines. You have been provided with Wind Turbine {Turb_ID}'s sensor data for Day {base_day}. 
The data is sampled every 10 minutes (144 rows total).

Columns: Tmstamp (time stamp), Wspd (wind speed in m/s), Wdir (The angle between the wind direction and the position of turbine nacelle in degrees), Etmp (Temperature of the surounding environment in celcius), Itmp (Temperature inside the turbine nacelle in celcius), Ndir (Nacelle direction, i.e., the yaw angle of the nacelle in degrees), Pab1 (Pitch angle of blade 1 in degrees), Pab2 (Pitch angle of blade 2 in degrees), Pab3 (Pitch angle of blade 3 in degrees), Prtv (Reactive power in kW), Patv (Active power in kW)

Input Data:
{data_str}

Task: Predict the 'Patv' (Active Power) for the next 24 hours (Day {base_day + 1}). Do not write code. Analyze the relationship between wind speed, temperature, and power in the provided data, and output your best estimate for the next 24 hours based on these trends. Return only a list of 144 numerical values separated by commas.
"""
    
    if PROVIDER == "claude":
        response = client.messages.create(
            model="claude-3-haiku-20240307",
            max_tokens=1024,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.content[0].text
    else:  # gemini
        response = client.generate_content(prompt)
        return response.text

In [22]:
df = pd.read_csv('wtbdata_245days.csv')
Turb_ID = 1
base_day = 2
output = call_llm(df, Turb_ID, base_day)
print(output)

1468.21,718.12,442.15,196.04,234.35,124.94,179.06,253.25,347.0,249.3,205.12,254.04,323.51,141.3,264.43,376.04,351.48,399.43,406.87,367.38,376.7,388.46,221.39,1014.57,1178.79,1001.72,550.82,364.21,607.99,628.92,690.13,984.27,1141.81,909.28,665.95,1101.66,1221.32,991.42,890.14,669.19,544.97,528.78,510.23,575.25,668.75,599.83,628.76,607.04,459.41,479.75,588.23,508.4,415.24,520.28,739.19,780.93,503.57,413.41,681.11,585.26,586.23,544.03,593.99,525.69,591.37,444.78,403.78,610.56,679.71,595.79,324.17,521.14,695.46,626.66,336.53,496.99,454.79,366.22,356.94,359.28,242.12,334.4,370.0,235.8,199.23,530.77,194.12,173.6,254.27,119.92,529.58,222.68,117.74,143.08,177.29,141.13,202.17,378.31,245.39,359.13,136.85,186.49,310.83,218.91,272.04,247.88,297.63,328.26,214.36,175.94,173.43,225.84,149.08,181.18,121.01,164.86,205.14,240.6,163.27,145.8,114.33,81.1,100.15,52.99,48.95,-1.29,0.0,21.09,275.76,826.09,1299.46,1513.79,1463.25,1433.66,1357.32,1524.53,1518.35,1549.98,1545.09,1516.64,1543.12,1549.74,1549.64

In [23]:
target_day = base_day + 1
actual_data = df[(df['TurbID'] == Turb_ID) & (df['Day'] == target_day)]

actual_values = actual_data['Patv'].dropna().tolist()
predicted_values = [float(val.strip()) for val in output.split(',')]

In [24]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Ensure both lists are the same length
min_len = min(len(actual_values), len(predicted_values))
y_true = actual_values[:min_len]
y_pred = predicted_values[:min_len]

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))

print(f"Mean Absolute Error: {mae:.2f} kW")
print(f"Root Mean Square Error: {rmse:.2f} kW")

Mean Absolute Error: 933.02 kW
Root Mean Square Error: 1017.72 kW


## Simple prompting is not ENOUGH

144 144
